# Fine-Tuning AI Search for E-commerce — Workshop Lab

**Welcome.** Over the next 60 minutes you'll go from a stock dense embedding model to a fine-tuned SPLADE + hybrid pipeline, on real Amazon ESCI data, and watch the numbers move.

### What you'll do

1. **CP1 — Baseline dense (12 min):** run ~10 hero queries against `all-MiniLM-L6-v2` and *feel* the failure modes. No aggregate metrics yet.
2. **CP2 — Fine-tuned SPLADE (20 min):** see what fine-tuned SPLADE buys you over baseline dense. First nDCG@10 / Recall@10 reveal across BM25 + baseline dense + fine-tuned SPLADE.
3. **CP3 — Hybrid fusion, two recipes (13 min):** fuse dense + BM25 *and* dense + fine-tuned SPLADE with RRF using Qdrant's Query API. Same call structure, two different sparse legs.
4. **Wrap (15 min):** full 2K-query eval with 95% CIs across all five approaches, the "where SPLADE breaks" honesty moment, takeaways.

### Live lab — no safety net

Every cell runs **live** against this VM. Nothing is precomputed — what you see is what the VM produced. If a cell errors, raise a hand — the instructor will troubleshoot.

## Pre-flight health check

One `[INFO]` debug line + three `[OK]` lines. If any of the `[OK]` lines fail, flag it before we start CP1.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve().parent  # kernel cwd is notebooks/ on the VM
DATA = REPO / "data"

sys.path.insert(0, str(REPO))

from qdrant_client import QdrantClient

from eval import SpladeEncoder
from scripts.setup_collections import (
    COLLECTION,
    DENSE_VECTOR_NAME,
    BM25_VECTOR_NAME,
    SPLADE_VECTOR_NAME,
    BM25_MODEL as BM25_MODEL_ID,
    DENSE_MODEL as DENSE_MODEL_ID,
    SPLADE_FINETUNED_MODEL_DEFAULT,
)

# throwaway client for the pre-flight checks; the Setup cell defines the canonical `client` handle
_client = QdrantClient(host="localhost", port=6333)
splade_encoder = SpladeEncoder(SPLADE_FINETUNED_MODEL_DEFAULT, device="cpu")


def _check(label, fn):
    try:
        fn()
        print(f"[OK]   {label}")
    except Exception as e:
        print(f"[FAIL] {label}: {e}")


def _assert_points():
    n = _client.get_collection(COLLECTION).points_count or 0
    assert n >= 10000, f"count {n} < 10K"


print(f"[INFO] repo root: {REPO}")
_check("Qdrant reachable at http://localhost:6333", lambda: _client.get_collections())
_check("products collection populated (10K)", _assert_points)
_check(
    f"SpladeEncoder loaded offline ({SPLADE_FINETUNED_MODEL_DEFAULT})",
    lambda: splade_encoder.encode(["ping"]),
)

## Setup

Load the curated query bundles, define the shared `retrieved_ids` helper, and pre-encode SPLADE for the eval subsets so the metric loops don't re-encode per call.

In [ ]:
import json

import pandas as pd
from IPython.display import display, Markdown
from qdrant_client import QdrantClient, models
from tqdm.auto import tqdm

from eval.metrics import ndcg_at_k, recall_at_k, bootstrap_ci
from eval.viewer import compare_results, inspect_sparse_vector, render_query_block
from eval.explain import explain_metric

client = QdrantClient(host="localhost", port=6333)


def retrieved_ids(points):
    """Return the ESCI product_id for each point, falling back to the raw UUID id.

    Points are upserted with ``uuid.uuid5(...)`` ids and the real ESCI
    ``product_id`` lives in the payload. Qrels are keyed by ``product_id``,
    so every eval needs to project payload-side ids before scoring.
    """
    return [p.payload.get("product_id", p.id) for p in points]


# Workshop data files. Missing files raise here -- setup failed, instructor handles.
hero_queries = json.loads((DATA / "hero_queries.json").read_text())
ood_queries  = json.loads((DATA / "ood_queries.json").read_text())
qrels        = json.loads((DATA / "qrels_hero.json").read_text())
subset_200   = json.loads((DATA / "eval_subset_200.json").read_text())
full_2k      = json.loads((DATA / "eval_full_2k.json").read_text())


In [ ]:
# Pre-encode SPLADE for the 200- and 2K-query subsets. Encoding once at startup
# is materially faster than re-encoding per call inside the metric loops.
# Batched in chunks of 32 so the encoder uses CPU parallelism instead of
# one-query-at-a-time. On a 4-vCPU VM the 2K loop is ~15-30s batched.
BATCH_SIZE = 32


def _batch_encode_splade(items, desc):
    out: dict = {}
    for i in tqdm(range(0, len(items), BATCH_SIZE), desc=desc, unit="batch"):
        batch = items[i:i + BATCH_SIZE]
        for item, (idx, vals) in zip(batch, splade_encoder.encode([it["query"] for it in batch])):
            out[item["query_id"]] = models.SparseVector(
                indices=list(map(int, idx)), values=list(map(float, vals)),
            )
    return out


splade_q_vecs_200 = _batch_encode_splade(subset_200, "pre-encoding 200-query SPLADE")
splade_q_vecs_2k  = _batch_encode_splade(full_2k, "pre-encoding 2K-query SPLADE")

display(Markdown(
    f"**Loaded {len(hero_queries)} hero queries, {len(ood_queries)} OOD queries, "
    f"{len(subset_200)} subset queries, {len(splade_q_vecs_2k)} 2K SPLADE pre-encodings.**"
))

---
## CP1 — Baseline dense (qualitative, 12 min)

**Goal:** see what `all-MiniLM-L6-v2` does on real e-commerce queries before we touch anything else.

**No aggregate metric yet.** Just look at the top-10 results. Each row is tagged with its ESCI grade (E/S/C/I) — watch for cases where the baseline returns a **Substitute** at rank 1 with the **Exact** buried deeper. The `65 lg tv` query (the same one from the hook slide) is one of the cleanest examples: baseline finds the first Exact only at rank 6.

We'll label the failure modes we spot after the queries run.

In [ ]:
def search_dense(query_text, k=10):
    """Baseline dense retrieval against the `products` collection's `dense` vector."""
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=query_text, model=DENSE_MODEL_ID),
        using=DENSE_VECTOR_NAME,
        limit=k,
        with_payload=True,
    ).points


def run_cp1_baseline():
    return {q["id"]: search_dense(q["text"]) for q in hero_queries}


cp1_results = run_cp1_baseline()

# CP1 is a 12-min checkpoint. Rendering all 10 heroes in full (top-10 + grades
# each) is too heavy a scan. Show three "focus" heroes in full so the room
# *feels* the failure modes, then summarize the other seven in a compact table.
#
# All three focus heroes are baseline-failures that fine-tuned SPLADE
# DEMONSTRABLY recovers into the top-3 (verified by scripts/verify_splade_wins.py),
# so the CP2 lift demo lands. Covers three distinct failure modes:
#   - specificity (size+brand)   -> ele_65_lg_tv           (matches the slide hook)
#   - identity (brand+model)     -> ele_apple_iphone_11_case
#   - specificity (quantity)     -> hom_11_piece_knife_set_without_block
cp1_focus_ids = [
    "ele_65_lg_tv",
    "ele_apple_iphone_11_case",
    "hom_11_piece_knife_set_without_block",
]

for qid in cp1_focus_ids:
    q = next(q for q in hero_queries if q["id"] == qid)
    display(Markdown(f"### `{q['id']}` — *{q['category']}* — **{q['text']}**"))
    compare_results(
        query=q["text"],
        models_results={"baseline_dense": cp1_results[q["id"]]},
        esci_qrels=qrels.get(q["id"]),
    )

# Compact summary for the other seven heroes: rank-1 title + ESCI grade.
_summary_rows = []
for q in hero_queries:
    if q["id"] in cp1_focus_ids:
        continue
    pts = cp1_results[q["id"]]
    if not pts:
        _summary_rows.append({
            "query_id": q["id"],
            "query_text": q["text"],
            "rank-1 title": "—",
            "rank-1 grade": "—",
        })
        continue
    top = pts[0]
    top_pid = top.payload.get("product_id", top.id)
    top_title = top.payload.get("product_title", top_pid)
    grade = (qrels.get(q["id"], {}) or {}).get(top_pid, "—")
    _summary_rows.append({
        "query_id": q["id"],
        "query_text": q["text"],
        "rank-1 title": str(top_title),
        "rank-1 grade": grade if grade != "—" else "—",
    })

_summary_df = pd.DataFrame(_summary_rows)
display(Markdown("#### Remaining seven heroes — rank-1 snapshot"))
display(_summary_df.style.hide(axis="index"))

### Label what you see

Take 2 minutes. For the three focus queries above, jot down which failure mode(s) jumped out:

- **Substitute-at-top** — wrong variant ("65 inch" query → 55 inch result; "11 piece" → 7 piece)
- **Complement-leak** — accessory instead of the product (iPhone query → iPhone case)
- **Missed-attribute** — color / size / brand / quantity ignored
- **Weak-exact-match** — near-paraphrase, not the exact SKU

| Query | Failure modes that jumped out |
|---|---|
| `ele_65_lg_tv` |  |
| `ele_apple_iphone_11_case` |  |
| `hom_11_piece_knife_set_without_block` |  |

We'll share out as a group before moving to CP2.

In [ ]:
# Scratch -- your notes / quick experiments


---
## CP2 — Fine-tuned SPLADE (first numbers, 20 min)

**Goal:** see what a SPLADE model fine-tuned on Amazon ESCI buys you over the dense baseline.

Same hero queries. Two retrievers side-by-side (baseline dense + fine-tuned SPLADE). Then we'll inspect a single fine-tuned sparse vector and finally show the first aggregate numbers on the 200-query slice.

In [ ]:
def search_bm25(query_text, k=10):
    """BM25 (Qdrant/bm25) retrieval. FastEmbed handles BM25, so the Document
    shortcut works here -- no need to pre-encode."""
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=query_text, model=BM25_MODEL_ID),
        using=BM25_VECTOR_NAME,
        limit=k,
        with_payload=True,
    ).points


def search_splade(query_text, k=10):
    """Fine-tuned SPLADE retrieval.

    FastEmbed does NOT support custom HF SPLADE checkpoints, so we encode the
    query with the pre-instantiated `splade_encoder` and pass a SparseVector
    to query_points.
    """
    idx, vals = splade_encoder.encode([query_text])[0]
    sv = models.SparseVector(
        indices=list(map(int, idx)), values=list(map(float, vals))
    )
    return client.query_points(
        collection_name=COLLECTION,
        query=sv,
        using=SPLADE_VECTOR_NAME,
        limit=k,
        with_payload=True,
    ).points


def run_splade_finetuned():
    return {q["id"]: search_splade(q["text"]) for q in hero_queries}


splade_ft = run_splade_finetuned()
display(Markdown(f"Got {len(splade_ft)} fine-tuned result sets."))


### Two-way side-by-side on two hero queries

We'll focus on two queries where the lift is most legible: an electronics SKU-variant query and an apparel attribute query. Watch the green Exact tags climb toward rank 1 as you go baseline → fine-tuned. Right after, we'll peek *inside* one SPLADE vector to see what the model actually encoded about the query.

In [ ]:
# CP2 focus heroes are a subset of CP1's three. Both are SPLADE wins (verified
# by scripts/verify_splade_wins.py: 65 lg tv climbs rank 6 -> 1, 11 piece knife
# set climbs rank 8 -> 3) so the lift is visually unambiguous in the room.
focus_ids = ["ele_65_lg_tv", "hom_11_piece_knife_set_without_block"]

for qid in focus_ids:
    q = next(q for q in hero_queries if q["id"] == qid)
    display(Markdown(f"### `{q['id']}` — **{q['text']}**"))
    compare_results(
        query=q["text"],
        models_results={
            "baseline_dense": cp1_results[qid],
            "splade_finetuned": splade_ft[qid],
        },
        esci_qrels=qrels.get(qid),
    )

### Peek inside a sparse vector (2 min)

Sparse is **interpretable** in a way dense isn't. Below we encode the **query** `"10 gallon fish tank"` with the fine-tuned SPLADE model and look at the top-weighted tokens. You should see the obvious literals — `tank`, `gallon`, `fish`, `10` — but also notice the **learned term expansions**: tokens like `tanks` (plural form), `capacity`, and `aquarium` that weren't in the query but show up because the model learned during fine-tuning that they're semantically related to fish-tank product titles. That's the lift over raw lexical matching.

*(The viewer filters pure punctuation and stopwords from the display by default — pass `skip_uninformative=False` to see the raw top-K.)*

In [ ]:
vocab = json.loads((DATA / "splade_vocab.json").read_text())
# splade_vocab.json keys come in as strings; viewer expects int keys.
vocab = {int(k): v for k, v in vocab.items()}

# Encode a query whose SPLADE expansion tells the cleanest story: 'aquarium'
# shows up in the top tokens for 'fish tank' even though it wasn't typed.
# That's the *learned* term expansion -- the thing fine-tuning teaches the
# sparse head to do beyond raw tokenization.
query_text = "10 gallon fish tank"
q_idx, q_vals = splade_encoder.encode([query_text])[0]
inspect_sparse_vector((q_idx, q_vals), vocab=vocab, top_k=12)

### First metric reveal — 200-query subset, 3-way

The 200-query subset is for **live feel**, not the headline claim. The full 2K with 95% CIs comes in the wrap.

Below: nDCG@10 and Recall@10 for **BM25** (lexical anchor) · **baseline dense** · **fine-tuned SPLADE**. Each metric has a plain-English translation underneath.

In [ ]:
def eval_subset(items, approaches):
    """Run each approach across every item and collect per-query nDCG@10 / Recall@10."""
    per_query = {name: {"ndcg": [], "recall": []} for name in approaches}
    for item in items:
        for name, fn in approaches.items():
            ids = retrieved_ids(fn(item["query"], item["query_id"]))
            per_query[name]["ndcg"].append(ndcg_at_k(ids, item["qrels"], k=10))
            per_query[name]["recall"].append(recall_at_k(ids, item["qrels"], k=10))
    return per_query


approaches_3way = {
    "BM25": lambda q, qid: client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=q, model=BM25_MODEL_ID),
        using=BM25_VECTOR_NAME, limit=10, with_payload=True,
    ).points,
    "baseline_dense": lambda q, qid: client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=q, model=DENSE_MODEL_ID),
        using=DENSE_VECTOR_NAME, limit=10, with_payload=True,
    ).points,
    "splade_finetuned": lambda q, qid: client.query_points(
        collection_name=COLLECTION,
        query=splade_q_vecs_200[qid],
        using=SPLADE_VECTOR_NAME, limit=10, with_payload=True,
    ).points,
}

# Cache to a notebook-scoped variable so the 5-way refresh in CP3 reuses it.
per_query_3way_200 = eval_subset(subset_200, approaches_3way)

rows = []
for approach in ("BM25", "baseline_dense", "splade_finetuned"):
    ndcgs = per_query_3way_200[approach]["ndcg"]
    recalls = per_query_3way_200[approach]["recall"]
    rows.append({
        "approach": approach,
        "nDCG@10": sum(ndcgs) / len(ndcgs),
        "Recall@10": sum(recalls) / len(recalls),
    })

df = pd.DataFrame(rows)
df["nDCG@10 (plain English)"] = df.apply(lambda r: explain_metric("nDCG@10", r["nDCG@10"]), axis=1)
df["Recall@10 (plain English)"] = df.apply(lambda r: explain_metric("Recall@10", r["Recall@10"]), axis=1)
display(Markdown("#### nDCG@10 / Recall@10 on the 200-query subset — *fast approximation, not the headline claim*"))
display(df.style.hide(axis="index").format({"nDCG@10": "{:.3f}", "Recall@10": "{:.3f}"}))


---
## CP3 — Hybrid fusion, two recipes (13 min)

**Goal:** combine dense with a sparse leg using **Reciprocal Rank Fusion** in a single Qdrant Query API call, and compare two recipes side by side:

- **Hybrid (D+BM25)** — dense + BM25. The classic production recipe.
- **Hybrid (D+SPLADE)** — dense + fine-tuned SPLADE. The workshop's pitch.

Same RRF call structure, two different sparse legs. Does fusing dense with BM25 match fusing dense with fine-tuned SPLADE on this corpus?

In [ ]:
def search_hybrid_splade(query_text, k=10, candidates=50):
    """Server-side RRF hybrid: dense + fine-tuned SPLADE.

    One `query_points` call with two `Prefetch` blocks (dense + splade_finetuned)
    and `FusionQuery(fusion=Fusion.RRF)`. No client-side fusion. This is the
    workshop's pitch.
    """
    splade_idx, splade_vals = splade_encoder.encode([query_text])[0]
    splade_sv = models.SparseVector(
        indices=list(map(int, splade_idx)), values=list(map(float, splade_vals))
    )
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query_text, model=DENSE_MODEL_ID),
                using=DENSE_VECTOR_NAME,
                limit=candidates,
            ),
            models.Prefetch(
                query=splade_sv,
                using=SPLADE_VECTOR_NAME,
                limit=candidates,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=k,
        with_payload=True,
    ).points


def search_hybrid_bm25(query_text, k=10, candidates=50):
    """Server-side RRF hybrid: dense + BM25.

    Same call structure as `search_hybrid_splade`, but the sparse leg is BM25
    instead of fine-tuned SPLADE. FastEmbed handles BM25, so we can pass
    `models.Document(...)` directly for that leg -- no precomputed cache needed.
    """
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query_text, model=DENSE_MODEL_ID),
                using=DENSE_VECTOR_NAME,
                limit=candidates,
            ),
            models.Prefetch(
                query=models.Document(text=query_text, model=BM25_MODEL_ID),
                using=BM25_VECTOR_NAME,
                limit=candidates,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=k,
        with_payload=True,
    ).points


hybrid_splade_results = {q["id"]: search_hybrid_splade(q["text"]) for q in hero_queries}
hybrid_bm25_results = {q["id"]: search_hybrid_bm25(q["text"]) for q in hero_queries}

# Continue the slide-hook through-line: 65 lg tv lands here so attendees see
# the same query from slide 2 -> CP1 -> CP2 -> CP3.
qid = "ele_65_lg_tv"
q = next(q for q in hero_queries if q["id"] == qid)
display(Markdown(f"### Two hybrid recipes on `{qid}` — **{q['text']}**"))
compare_results(
    query=q["text"],
    models_results={
        "splade_finetuned": splade_ft[qid],
        "hybrid_bm25": hybrid_bm25_results[qid],
        "hybrid_splade": hybrid_splade_results[qid],
    },
    esci_qrels=qrels.get(qid),
)

### 5-way metric refresh on the 200-query subset

Same caveat as CP2: this is the in-room feel, the full 2K with CIs is in the wrap.

Watch the two `Hybrid (...)` rows — same RRF structure, different sparse leg. Does dense + BM25 (the classic) match dense + fine-tuned SPLADE (the workshop's pitch)?

In [ ]:
def eval_hybrid_splade_on_subset():
    ndcgs, recalls = [], []
    for item in subset_200:
        pts = client.query_points(
            collection_name=COLLECTION,
            prefetch=[
                models.Prefetch(
                    query=models.Document(text=item["query"], model=DENSE_MODEL_ID),
                    using=DENSE_VECTOR_NAME,
                    limit=50,
                ),
                models.Prefetch(
                    query=splade_q_vecs_200[item["query_id"]],
                    using=SPLADE_VECTOR_NAME,
                    limit=50,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=10,
            with_payload=True,
        ).points
        ids = retrieved_ids(pts)
        ndcgs.append(ndcg_at_k(ids, item["qrels"], k=10))
        recalls.append(recall_at_k(ids, item["qrels"], k=10))
    return {"ndcg": ndcgs, "recall": recalls}


def eval_hybrid_bm25_on_subset():
    ndcgs, recalls = [], []
    for item in subset_200:
        pts = client.query_points(
            collection_name=COLLECTION,
            prefetch=[
                models.Prefetch(
                    query=models.Document(text=item["query"], model=DENSE_MODEL_ID),
                    using=DENSE_VECTOR_NAME,
                    limit=50,
                ),
                models.Prefetch(
                    query=models.Document(text=item["query"], model=BM25_MODEL_ID),
                    using=BM25_VECTOR_NAME,
                    limit=50,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=10,
            with_payload=True,
        ).points
        ids = retrieved_ids(pts)
        ndcgs.append(ndcg_at_k(ids, item["qrels"], k=10))
        recalls.append(recall_at_k(ids, item["qrels"], k=10))
    return {"ndcg": ndcgs, "recall": recalls}


# Reuse the cached 3-way per-query scores from CP2; we only run the two
# hybrid recipes here. Appending to the same dict keeps the LINEUP_200
# render below trivial.
per_query_3way_200["hybrid_bm25"] = eval_hybrid_bm25_on_subset()
per_query_3way_200["hybrid_splade"] = eval_hybrid_splade_on_subset()

# Display labels for the 5-way table, ordered: lexical anchor, dense, sparse-learned, two hybrids.
LINEUP_200 = [
    ("BM25",             "BM25"),
    ("baseline_dense",   "Dense"),
    ("splade_finetuned", "SPLADE"),
    ("hybrid_bm25",      "Hybrid (D+BM25)"),
    ("hybrid_splade",    "Hybrid (D+SPLADE)"),
]

rows5 = []
for key, label in LINEUP_200:
    ndcgs = per_query_3way_200[key]["ndcg"]
    recalls = per_query_3way_200[key]["recall"]
    rows5.append({
        "approach": label,
        "nDCG@10": sum(ndcgs) / len(ndcgs),
        "Recall@10": sum(recalls) / len(recalls),
    })

df5 = pd.DataFrame(rows5)
df5["nDCG@10 (plain English)"] = df5.apply(lambda r: explain_metric("nDCG@10", r["nDCG@10"]), axis=1)
df5["Recall@10 (plain English)"] = df5.apply(lambda r: explain_metric("Recall@10", r["Recall@10"]), axis=1)
display(Markdown("#### 5-way on the 200-query subset"))
display(df5.style.hide(axis="index").format({"nDCG@10": "{:.3f}", "Recall@10": "{:.3f}"}))


---
## Wrap — the headline numbers, where it breaks, and your turn (15 min)

### The authoritative claim — full 2K eval with 95% CIs

Computed **live** on this VM against the full 2K ESCI test split. **This is the number we stand behind.** Bootstrap 95% CIs come from 1,000 resamples per approach.

Running the full 2K eval live takes 2–6 min on the VM (exact range CPU-dependent, verified at pilot). Watch the progress bar; the headline table renders underneath when it's done.

Plain-English translations sit under each quality column. The table compares all five approaches: BM25, Dense, SPLADE, Hybrid (D+BM25), Hybrid (D+SPLADE).

In [ ]:
def search_bm25_2k(q, qid):
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=q, model=BM25_MODEL_ID),
        using=BM25_VECTOR_NAME, limit=10, with_payload=True,
    ).points


def search_dense_2k(q, qid):
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=q, model=DENSE_MODEL_ID),
        using=DENSE_VECTOR_NAME, limit=10, with_payload=True,
    ).points


def search_splade_2k(q, qid):
    return client.query_points(
        collection_name=COLLECTION,
        query=splade_q_vecs_2k[qid],
        using=SPLADE_VECTOR_NAME, limit=10, with_payload=True,
    ).points


def search_hybrid_bm25_2k(q, qid):
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=models.Document(text=q, model=DENSE_MODEL_ID),
                            using=DENSE_VECTOR_NAME, limit=50),
            models.Prefetch(query=models.Document(text=q, model=BM25_MODEL_ID),
                            using=BM25_VECTOR_NAME, limit=50),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=10, with_payload=True,
    ).points


def search_hybrid_splade_2k(q, qid):
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=models.Document(text=q, model=DENSE_MODEL_ID),
                            using=DENSE_VECTOR_NAME, limit=50),
            models.Prefetch(query=splade_q_vecs_2k[qid],
                            using=SPLADE_VECTOR_NAME, limit=50),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=10, with_payload=True,
    ).points


# Display labels feed the headline table directly; internal keys keep code paths readable.
retrievers = {
    "BM25":              search_bm25_2k,
    "Dense":             search_dense_2k,
    "SPLADE":            search_splade_2k,
    "Hybrid (D+BM25)":   search_hybrid_bm25_2k,
    "Hybrid (D+SPLADE)": search_hybrid_splade_2k,
}

per_query = {name: {"ndcg": [], "recall": []} for name in retrievers}

for item in tqdm(full_2k, desc="2K eval", unit="q"):
    q_text, q_rels, qid = item["query"], item["qrels"], item["query_id"]
    for name, fn in retrievers.items():
        ids = retrieved_ids(fn(q_text, qid))
        per_query[name]["ndcg"].append(ndcg_at_k(ids, q_rels, k=10))
        per_query[name]["recall"].append(recall_at_k(ids, q_rels, k=10))

full_rows = []
for name in retrievers:
    ndcg_mean, ndcg_lo, ndcg_hi = bootstrap_ci(per_query[name]["ndcg"], n_bootstrap=1000)
    rec_mean,  rec_lo,  rec_hi  = bootstrap_ci(per_query[name]["recall"], n_bootstrap=1000)
    full_rows.append({
        "approach":   name,
        "nDCG@10":    ndcg_mean,
        "nDCG@10_lo": ndcg_lo,
        "nDCG@10_hi": ndcg_hi,
        "Recall@10":    rec_mean,
        "Recall@10_lo": rec_lo,
        "Recall@10_hi": rec_hi,
    })

display(Markdown(f"**Live 2K eval complete** — {len(full_2k)} queries, {len(retrievers)} retrievers, bootstrap CIs from 1,000 resamples."))


In [ ]:
full_df = pd.DataFrame(full_rows)


def fmt_ci(row, base):
    return f"{row[base]:.3f}  [{row[base + '_lo']:.3f}, {row[base + '_hi']:.3f}]"


full_df["nDCG@10 (95% CI)"]   = full_df.apply(lambda r: fmt_ci(r, "nDCG@10"), axis=1)
full_df["Recall@10 (95% CI)"] = full_df.apply(lambda r: fmt_ci(r, "Recall@10"), axis=1)
full_df["nDCG@10 (plain English)"]   = full_df.apply(lambda r: explain_metric("nDCG@10", r["nDCG@10"]), axis=1)
full_df["Recall@10 (plain English)"] = full_df.apply(lambda r: explain_metric("Recall@10", r["Recall@10"]), axis=1)

display(Markdown("#### Full 2K ESCI test set — *the headline claim*"))
cols = [
    "approach",
    "nDCG@10 (95% CI)",
    "nDCG@10 (plain English)",
    "Recall@10 (95% CI)",
    "Recall@10 (plain English)",
]
display(full_df[cols].style.hide(axis="index"))


### Stretch — your turn

If we have time, type a query you actually care about. We'll fire it through all five retrievers and look at the results together.

In [ ]:
your_query = "ergonomic standing desk converter dual monitor"  # <-- edit me

def your_turn(q):
    return {
        "BM25":              search_bm25(q),
        "Dense":             search_dense(q),
        "SPLADE":            search_splade(q),
        "Hybrid (D+BM25)":   search_hybrid_bm25(q),
        "Hybrid (D+SPLADE)": search_hybrid_splade(q),
    }


results = your_turn(your_query)
compare_results(query=your_query, models_results=results, esci_qrels=None)


### Where this model is the wrong tool

Fine-tuned SPLADE is **specialized for product retrieval**. Throw a general-knowledge query at it and it'll return product results — because that's what its corpus and training distribution contain. This isn't "the model is broken"; it's "the model is the wrong tool for non-shopping intent."

In production, this is solved by **query routing** (classify shopping vs general intent upstream) or a **cross-encoder rerank** stage that can recognize off-distribution queries. Don't expect a fine-tuned retrieval model to do its own intent detection.

The five OOD results below each illustrate the same point: shopping corpus + shopping training = shopping answers, regardless of what was asked.


In [ ]:
for i, q in enumerate(ood_queries):
    results = search_splade(q["text"])
    render_query_block(q["text"], results, expanded=(i == 0))


### Q&A

Open floor. A few prompts to seed it:

- **"What about cross-encoder rerankers?"** — That's the next layer of the production stack. Hybrid retrieval → cross-encoder rerank on the top-100 → final top-10. Deferred to a follow-up workshop.
- **"How big does my training set need to be?"** — The model you used today was trained on ~100K query-product pairs in ~6 min on an A100. Diminishing returns kick in well before 1M.
- **"Can I just keep using BM25?"** — Look at the gap on the wrap table. BM25 is a hard floor, not a ceiling.
- **"What if my queries shift over time?"** — Schedule re-tuning. The fine-tuning cost is low. Catastrophic forgetting risk is real if you cross domains — see the self-study notebook.

---
## Appendix — stretch goals for fast finishers

### Alternate fusion strategies for CP3 hybrid

Qdrant's `Fusion.RRF` has **no native per-leg weights** — RRF combines lists by rank position, not score. In production the two real-world dials are:

1. **Swap the fusion strategy.** Use `models.Fusion.DBSF` (Distribution-Based Score Fusion). DBSF normalizes each retriever's score distribution before combining, instead of voting on rank position. Behaves differently when one retriever's score distribution is much wider than the other's.
2. **Imbalance the per-leg `Prefetch.limit`.** RRF treats both lists equally given the same length. Make one prefetch's limit larger than the other (e.g. `dense=10, splade=80`) — you've effectively biased fusion toward SPLADE without changing the fusion math.

The code stub below mirrors `search_hybrid_splade` but uses `Fusion.DBSF`. Copy-paste, edit, and run.

In [ ]:
def search_hybrid_dbsf(query_text, k=10, candidates=50):
    """Server-side DBSF hybrid -- identical to search_hybrid_splade but with Fusion.DBSF."""
    splade_idx, splade_vals = splade_encoder.encode([query_text])[0]
    splade_sv = models.SparseVector(
        indices=list(map(int, splade_idx)), values=list(map(float, splade_vals))
    )
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query_text, model=DENSE_MODEL_ID),
                using=DENSE_VECTOR_NAME,
                limit=candidates,
            ),
            models.Prefetch(
                query=splade_sv,
                using=SPLADE_VECTOR_NAME,
                limit=candidates,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.DBSF),
        limit=k,
        with_payload=True,
    ).points


# Same through-line hero as CP3.
_qid = "ele_65_lg_tv"
_q = next(q for q in hero_queries if q["id"] == _qid)
compare_results(
    query=_q["text"],
    models_results={
        "hybrid_splade": hybrid_splade_results[_qid],
        "hybrid_dbsf": search_hybrid_dbsf(_q["text"]),
    },
    esci_qrels=qrels.get(_qid),
)

### Takeaway links

- **Article series — Sparse Embeddings for E-commerce (part 1)** — https://qdrant.tech/articles/sparse-embeddings-ecommerce-part-1/
- **Fine-tuned model on HuggingFace** — https://huggingface.co/thierrydamiba/splade-ecommerce-esci
- **Live 2K eval** — runs inside this notebook (see the "Wrap" section above), no separate script
- **Collection setup (replicate on a different VM)** — `scripts/setup_collections.py`
- **Training notebook (self-study)** — `notebooks/splade_training.ipynb` in this repo
- **Amazon ESCI dataset** — https://github.com/amazon-science/esci-data

Thanks for coming. Go fine-tune something.